# Baseline Machine Learning for Store Sales Forecasting



## 1. Setup and imports

This section loads the standard scientific Python stack and points the notebook to the project root so the `src` package can be imported cleanly.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Load and preprocess the retail panel

The preprocessing module handles raw loading, safe merges, holiday expansion, oil alignment, transaction cleaning, and train-based outlier capping.

Important design choice: the preprocessing is fit on training data only, so the validation and test sets do not leak future information into the cleaning step.

In [6]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "processed"

train_df = pd.read_parquet(data_dir / "train.parquet")
val_df = pd.read_parquet(data_dir / "val.parquet")
test_df = pd.read_parquet(data_dir / "test.parquet")

print(f"Loaded from: {data_dir}")
print(f"train: {train_df.shape} | val: {val_df.shape} | test: {test_df.shape}")

Loaded from: c:\Users\MONTAHA\Desktop\Ai\forecasting-ml-system\data\processed
train: (2918916, 23) | val: (53460, 23) | test: (28512, 23)


In [7]:
target_col = "sales"
drop_columns = ["sales", "date"]

X_train = train_df.drop(columns=drop_columns, errors="ignore")
y_train = train_df[target_col].astype(float)

X_val = val_df.drop(columns=drop_columns, errors="ignore")
y_val = val_df[target_col].astype(float)

X_test = test_df.drop(columns=drop_columns, errors="ignore")
y_test = test_df[target_col].astype(float)

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

X_train: (2918916, 21), X_val: (53460, 21), X_test: (28512, 21)


## 3. Create leakage-safe features

We now transform the merged panel into model-ready features.

The key rule is that every historical signal used by the model must come from the past only. That means lag features and rolling statistics are shifted before aggregation.

In [8]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))

Numeric: 14
Categorical: 7


### Notes on the feature design

- `sales_lag_*` captures short and medium-term demand memory.
- rolling mean/std/min/max helps the model understand local trend and volatility.
- calendar features capture weekly and monthly seasonality.
- promotion, transaction, and oil features are kept only as past-based signals.

This is a good baseline structure because it is simple, robust, and explainable.

In [9]:
date_col = "date"

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(
        f"{split_name}: {split_df.shape} | "
        f"{split_df[date_col].min()} -> {split_df[date_col].max()}"
    )

train: (2918916, 23) | 2013-01-01 00:00:00 -> 2017-06-30 00:00:00
val: (53460, 23) | 2017-07-01 00:00:00 -> 2017-07-30 00:00:00
test: (28512, 23) | 2017-07-31 00:00:00 -> 2017-08-15 00:00:00


## 4. Define baseline target and features

For a first baseline, we predict daily sales directly. The feature set includes time-derived variables, lagged demand, rolling statistics, promotions, transactions, oil, and categorical store/family identifiers.

In [10]:
drop_columns = [
    'sales',
    'date',
]

X_train = train_df.drop(columns=[c for c in drop_columns if c in train_df.columns])
y_train = train_df[target_col].astype(float)

X_val = val_df.drop(columns=[c for c in drop_columns if c in val_df.columns])
y_val = val_df[target_col].astype(float)

X_test = test_df.drop(columns=[c for c in drop_columns if c in test_df.columns])
y_test = test_df[target_col].astype(float)

numeric_features = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')
print('Categorical sample:', categorical_features[:10])

Numeric features: 14
Categorical features: 7
Categorical sample: ['family', 'city', 'state', 'type_x', 'type_y', 'locale', 'locale_name']


## 5. Model 1: naive seasonal baseline

Before training any machine learning model, we evaluate a simple naive baseline.

Here the forecast is the 7-day lagged sales value. This is a strong sanity check because time-series models should beat it clearly to be considered useful.

In [11]:
def rmsle(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true), 0)
    y_pred = np.maximum(np.asarray(y_pred), 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

season = 7

naive_val = val_df["sales"].shift(season)
naive_test = test_df["sales"].shift(season)

naive_val = naive_val.fillna(train_df["sales"].mean())
naive_test = naive_test.fillna(train_df["sales"].mean())

naive_metrics = {
    "validation_mae": mean_absolute_error(y_val, naive_val),
    "validation_rmse": np.sqrt(mean_squared_error(y_val, naive_val)),
    "validation_rmsle": rmsle(y_val, naive_val),
    "test_mae": mean_absolute_error(y_test, naive_test),
    "test_rmse": np.sqrt(mean_squared_error(y_test, naive_test)),
    "test_rmsle": rmsle(y_test, naive_test),
}

pd.Series(naive_metrics).to_frame("value")

,value
validation_mae,881.522478
validation_rmse,1951.866412
validation_rmsle,3.920209
test_mae,835.520492
test_rmse,1803.552621
test_rmsle,3.859030


## 6. Model 2: ridge regression baseline

A regularized linear model is a good baseline for retail forecasting because it is fast, stable, and easy to interpret.

We use one-hot encoding for categorical variables and standardization for numeric variables. Missing numeric values are imputed with the median, and missing categorical values with the most frequent category.

In [12]:
target_col = "sales"
drop_cols = ["sales", "date"]

X_train = train_df.drop(columns=drop_cols, errors="ignore")
y_train = train_df[target_col]

X_val = val_df.drop(columns=drop_cols, errors="ignore")
y_val = val_df[target_col]

X_test = test_df.drop(columns=drop_cols, errors="ignore")
y_test = test_df[target_col]

# Encode categorical variables
X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns to training set
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train, y_train)

val_pred = ridge_model.predict(X_val)
test_pred = ridge_model.predict(X_test)

def rmsle(y_true, y_pred):
    y_true = np.maximum(y_true, 0)
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

ridge_metrics = {
    "validation_mae": mean_absolute_error(y_val, val_pred),
    "validation_rmse": np.sqrt(mean_squared_error(y_val, val_pred)),
    "validation_rmsle": rmsle(y_val, val_pred),
    "test_mae": mean_absolute_error(y_test, test_pred),
    "test_rmse": np.sqrt(mean_squared_error(y_test, test_pred)),
    "test_rmsle": rmsle(y_test, test_pred),
}

pd.Series(ridge_metrics).to_frame("ridge_baseline")

,ridge_baseline
validation_mae,316.502152
validation_rmse,794.855894
validation_rmsle,2.051011
test_mae,295.428781
test_rmse,698.862646
test_rmsle,2.022613


### Baseline comparison

This table compares the naive seasonal forecast with the ridge regression baseline. In practice, the ridge model should outperform the naive method if the feature engineering is effective.

In [13]:
comparison = pd.DataFrame({
    'naive_validation_rmsle': [naive_metrics['validation_rmsle']],
    'ridge_validation_rmsle': [ridge_metrics['validation_rmsle']],
    'naive_test_rmsle': [naive_metrics['test_rmsle']],
    'ridge_test_rmsle': [ridge_metrics['test_rmsle']],
})
comparison.T.rename(columns={0: 'score'})

,score
naive_validation_rmsle,3.920209
ridge_validation_rmsle,2.051011
naive_test_rmsle,3.859030
ridge_test_rmsle,2.022613


## 7. Model 3: random forest baseline

Tree-based models can capture nonlinear interactions between promotions, seasonality, and lag features.

This model is usually heavier than ridge regression, but it can be useful as a stronger classical benchmark.

In [14]:
rf_model = RandomForestRegressor(
    n_estimators=80,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)


# TRAIN

rf_model.fit(X_train, y_train)

# PREDICTIONS


rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)


# METRICS
def rmsle(y_true, y_pred):
    y_true = np.maximum(y_true, 0)
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

rf_metrics = {
    "validation_mae": mean_absolute_error(y_val, rf_val_pred),
    "validation_rmse": np.sqrt(mean_squared_error(y_val, rf_val_pred)),
    "validation_rmsle": rmsle(y_val, rf_val_pred),

    "test_mae": mean_absolute_error(y_test, rf_test_pred),
    "test_rmse": np.sqrt(mean_squared_error(y_test, rf_test_pred)),
    "test_rmsle": rmsle(y_test, rf_test_pred),
}

pd.Series(rf_metrics).to_frame("rf_baseline")

,rf_baseline
validation_mae,74.382722
validation_rmse,223.202009
validation_rmsle,1.208159
test_mae,81.943304
test_rmse,242.711385
test_rmsle,1.214093


## 8. Final interpretation

Use the best-performing validation model for any final report, then re-evaluate once on the test set.

For the Kaggle project, the next step after this notebook is usually to move from classical baselines to a sequence model such as LSTM or Transformer, keeping the same leakage-safe data pipeline.

In [15]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

ridge_val_pred = ridge_model.predict(X_val)
ridge_test_pred = ridge_model.predict(X_test)

def rmsle(y_true, y_pred):
    y_true = np.maximum(y_true, 0)
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

ridge_metrics = {
    "validation_rmsle": rmsle(y_val, ridge_val_pred),
    "test_rmsle": rmsle(y_test, ridge_test_pred)
}

In [16]:
results = pd.DataFrame([
    {
        "model": "naive",
        "validation_rmsle": naive_metrics["validation_rmsle"],
        "test_rmsle": naive_metrics["test_rmsle"]
    },
    {
        "model": "ridge",
        "validation_rmsle": ridge_metrics["validation_rmsle"],
        "test_rmsle": ridge_metrics["test_rmsle"]
    },
    {
        "model": "random_forest",
        "validation_rmsle": rf_metrics["validation_rmsle"],
        "test_rmsle": rf_metrics["test_rmsle"]
    },
])

results.sort_values("validation_rmsle")

,model,validation_rmsle,test_rmsle
2,random_forest,1.208159,1.214093
1,ridge,2.051011,2.022613
0,naive,3.920209,3.859030
